# 🏥 PCOS Risk Screening - Complete ML Pipeline
## Early PCOS Risk Screening using Physiological Parameters

**A production-ready machine learning pipeline for biomedical applications**

This notebook implements a complete ML workflow:
- Data loading and exploration
- Feature normalization
- Synthetic data augmentation (SMOTE + Gaussian noise)
- Multi-model training (Logistic Regression, Random Forest, XGBoost)
- Comprehensive evaluation and visualization
- Model comparison and selection

## Step 1: Install Dependencies & Import Libraries

In [ ]:
# Install required packages
!pip install scikit-learn xgboost imbalanced-learn pandas numpy matplotlib seaborn joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, auc, classification_report
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

# Configure visualization
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n" + "="*100)
print("PCOS RISK SCREENING - COMPLETE ML PIPELINE")
print("="*100)
print("\n✓ All libraries imported successfully!")

## Step 2: Create Sample Dataset

**Note:** You can replace this with your own CSV file using:
```python
df = pd.read_csv('your_file.csv')
```

In [ ]:
print("\n[STEP 1: LOADING DATA]")
print("-" * 100)

# Create sample dataset
data = {
    'HRV': [42.5, 35.8, 51.2, 38.5, 48.3, 32.1, 55.7, 36.9, 44.2, 33.4,
            49.8, 37.6, 52.1, 34.9, 46.5, 39.2, 54.3, 35.1, 43.8, 31.5,
            50.6, 36.4, 45.9, 38.1, 53.5, 33.7, 41.2, 37.8, 48.9, 34.2],
    'Heart_Rate': [68, 75, 62, 72, 65, 80, 58, 76, 70, 79,
                   64, 74, 60, 77, 67, 73, 61, 78, 69, 82,
                   63, 75, 66, 76, 59, 80, 71, 74, 65, 79],
    'RR_Mean': [14.2, 15.1, 13.8, 14.9, 14.5, 15.8, 13.2, 15.3, 14.6, 15.6,
                14.0, 15.2, 13.9, 15.4, 14.4, 15.0, 13.5, 15.7, 14.7, 16.0,
                13.7, 15.1, 14.3, 15.5, 13.6, 15.9, 14.8, 15.2, 14.1, 15.8],
    'BMI': [22.5, 24.3, 20.1, 26.8, 21.8, 28.5, 19.5, 27.2, 22.8, 29.1,
            21.2, 25.9, 20.8, 27.6, 23.1, 26.2, 20.5, 28.2, 22.4, 29.8,
            21.5, 25.4, 23.5, 27.0, 20.2, 28.8, 22.2, 26.5, 21.0, 28.1],
    'BMR': [1650, 1580, 1720, 1520, 1690, 1480, 1780, 1510, 1640, 1450,
            1710, 1550, 1750, 1490, 1670, 1530, 1770, 1460, 1660, 1420,
            1700, 1560, 1680, 1500, 1760, 1440, 1650, 1520, 1720, 1470],
    'Systolic_BP': [118, 125, 110, 132, 115, 138, 108, 128, 120, 135,
                    112, 130, 109, 126, 117, 131, 111, 136, 119, 140,
                    114, 127, 118, 129, 110, 137, 121, 132, 113, 134],
    'Diastolic_BP': [76, 82, 70, 85, 74, 88, 68, 83, 77, 87,
                     72, 84, 69, 82, 75, 85, 71, 86, 77, 89,
                     73, 81, 76, 84, 70, 87, 78, 86, 72, 85],
    'Temperature': [37.1, 37.2, 36.9, 37.3, 37.0, 37.4, 36.8, 37.2, 37.1, 37.5,
                    37.0, 37.3, 36.9, 37.2, 37.1, 37.4, 36.8, 37.5, 37.0, 37.6,
                    37.1, 37.2, 37.1, 37.3, 36.9, 37.5, 37.0, 37.4, 37.1, 37.4],
    'Age': [22, 25, 21, 28, 23, 30, 20, 26, 24, 29,
            22, 27, 21, 28, 25, 26, 23, 30, 24, 32,
            22, 27, 25, 29, 21, 31, 23, 28, 23, 30],
    'PCOS': [0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
             0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
             0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
}

df = pd.DataFrame(data)
print(f"✓ Dataset loaded successfully!")
print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nSummary Statistics:")
print(df.describe())

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
print("\n[STEP 2: EXPLORATORY DATA ANALYSIS]")
print("-" * 100)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('PCOS Dataset - Exploratory Data Analysis', fontsize=16, fontweight='bold')

# Class Distribution
class_counts = df['PCOS'].value_counts()
axes[0, 0].bar(['No PCOS (0)', 'PCOS (1)'], class_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0, 0].set_title('Class Distribution', fontweight='bold')
axes[0, 0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0, 0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

print(f"✓ Class Distribution:")
print(f"  - PCOS=0 (Negative): {class_counts[0]} ({class_counts[0]/len(df)*100:.1f}%)")
print(f"  - PCOS=1 (Positive): {class_counts[1]} ({class_counts[1]/len(df)*100:.1f}%)")

# Correlation Heatmap
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[0, 1], cbar_kws={'label': 'Correlation'})
axes[0, 1].set_title('Feature Correlation Heatmap', fontweight='bold')

# Age Distribution
df.boxplot(column='Age', by='PCOS', ax=axes[1, 0])
axes[1, 0].set_title('Age Distribution by PCOS Status', fontweight='bold')
axes[1, 0].set_xlabel('PCOS Status')
axes[1, 0].set_ylabel('Age (years)')

# BMI Distribution
df.boxplot(column='BMI', by='PCOS', ax=axes[1, 1])
axes[1, 1].set_title('BMI Distribution by PCOS Status', fontweight='bold')
axes[1, 1].set_xlabel('PCOS Status')
axes[1, 1].set_ylabel('BMI (kg/m²)')

plt.tight_layout()
plt.show()
print("✓ EDA visualization complete!")

## Step 4: Feature Preparation & Normalization

In [ ]:
print("\n[STEP 3: FEATURE PREPARATION & NORMALIZATION]")
print("-" * 100)

# Separate features and target
feature_names = [col for col in df.columns if col != 'PCOS']
X = df[feature_names].values
y = df['PCOS'].values

print(f"✓ Feature matrix shape: {X.shape}")
print(f"✓ Target distribution: {np.bincount(y.astype(int))}")

# Normalize features
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)

print(f"✓ Features normalized using StandardScaler")
print(f"  - Mean (should be ~0): {X_normalized.mean(axis=0).round(3)}")
print(f"  - Std (should be ~1): {X_normalized.std(axis=0).round(3)}")

## Step 5: Synthetic Data Augmentation (SMOTE + Gaussian Noise)

⚠️ **IMPORTANT:** Synthetic data is for training robustness. Generated samples are NOT actual patients and should NOT be used for clinical validation without real data.

In [ ]:
print("\n[STEP 4: SYNTHETIC DATA AUGMENTATION]")
print("-" * 100)

print("\n⚠️  IMPORTANT: Synthetic data used for training robustness only!")
print("   Generated samples should NOT be used for clinical validation without real data.\n")

# Apply SMOTE
print("1. Applying SMOTE (Synthetic Minority Over-sampling Technique)...")
smote = SMOTE(random_state=42, k_neighbors=3)
X_smote, y_smote = smote.fit_resample(X_normalized, y)
print(f"   ✓ After SMOTE:")
print(f"     - New shape: {X_smote.shape}")
print(f"     - Class distribution: {np.bincount(y_smote.astype(int))}")

# Gaussian Noise Augmentation
print("\n2. Applying Gaussian Noise Augmentation...")
noise_scales = {
    'HRV': 2.0,
    'Heart_Rate': 3.0,
    'RR_Mean': 0.3,
    'BMI': 0.5,
    'BMR': 30.0,
    'Systolic_BP': 3.0,
    'Diastolic_BP': 2.0,
    'Temperature': 0.2,
    'Age': 0.0
}

X_augmented_list = [X_smote.copy()]
y_augmented_list = [y_smote.copy()]

for noise_factor in [0.5, 0.25]:
    X_noise = X_smote.copy()
    for feature_idx, feature_name in enumerate(feature_names):
        if feature_name in noise_scales:
            noise = np.random.normal(0, noise_scales[feature_name] * noise_factor, X_noise.shape[0])
            X_noise[:, feature_idx] += noise
    X_augmented_list.append(X_noise)
    y_augmented_list.append(y_smote.copy())

X_augmented = np.vstack(X_augmented_list)
y_augmented = np.hstack(y_augmented_list)

print(f"   ✓ After Gaussian Noise Augmentation:")
print(f"     - New shape: {X_augmented.shape}")
print(f"     - Class distribution: {np.bincount(y_augmented.astype(int))}")

## Step 6: Train-Test Split

In [ ]:
print("\n[STEP 5: TRAIN-TEST SPLIT]")
print("-" * 100)

X_train, X_test, y_train, y_test = train_test_split(
    X_augmented, y_augmented, test_size=0.2, random_state=42, stratify=y_augmented
)

print(f"✓ Data split successfully:")
print(f"  - Training set: {X_train.shape[0]} samples")
print(f"  - Test set: {X_test.shape[0]} samples")
print(f"  - Train class distribution: {np.bincount(y_train.astype(int))}")
print(f"  - Test class distribution: {np.bincount(y_test.astype(int))}")

## Step 7: Model Training

Training three different models:
1. **Logistic Regression** - Fast, interpretable baseline
2. **Random Forest** - Ensemble method with feature importance
3. **XGBoost** - Gradient boosting, often best performance

In [ ]:
print("\n[STEP 6: MODEL TRAINING]")
print("-" * 100)

print("\n1. Training Logistic Regression...")
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
print("   ✓ Logistic Regression trained")

print("\n2. Training Random Forest Classifier...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print("   ✓ Random Forest trained")

print("\n3. Training XGBoost Classifier...")
xgb = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbosity=0)
xgb.fit(X_train, y_train, verbose=False)
print("   ✓ XGBoost trained")

models = {
    'Logistic Regression': lr,
    'Random Forest': rf,
    'XGBoost': xgb
}

print("\n✓ All models trained successfully!")

## Step 8: Model Evaluation & Comparison

In [ ]:
print("\n[STEP 7: MODEL EVALUATION & COMPARISON]")
print("-" * 100)

results = []
predictions = {}
probabilities = {}

for model_name, model in models.items():
    print(f"\nEvaluating {model_name}...")
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    predictions[model_name] = y_pred
    probabilities[model_name] = y_pred_proba
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc
    })
    
    print(f"  - Accuracy:  {accuracy:.4f}")
    print(f"  - Precision: {precision:.4f}")
    print(f"  - Recall:    {recall:.4f}")
    print(f"  - F1-Score:  {f1:.4f}")
    print(f"  - ROC-AUC:   {roc_auc:.4f}")
    
    cm = confusion_matrix(y_test, y_pred)
    print(f"\n  Confusion Matrix:")
    print(f"    True Negatives:  {cm[0,0]}, False Positives: {cm[0,1]}")
    print(f"    False Negatives: {cm[1,0]}, True Positives:  {cm[1,1]}")

results_df = pd.DataFrame(results)
print("\n" + "="*100)
print("MODEL COMPARISON SUMMARY")
print("="*100)
print(results_df.to_string(index=False))

## Step 9: Visualize Results

In [ ]:
print("\n[STEP 8: VISUALIZATION OF RESULTS]")
print("-" * 100)

# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices - Test Set', fontsize=14, fontweight='bold')

for idx, (model_name, model) in enumerate(models.items()):
    cm = confusion_matrix(y_test, predictions[model_name])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
               xticklabels=['No PCOS', 'PCOS'],
               yticklabels=['No PCOS', 'PCOS'],
               cbar=False)
    axes[idx].set_title(model_name, fontweight='bold')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()
print("✓ Confusion matrices visualized")

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(10, 8))

colors = ['#3498db', '#e74c3c', '#2ecc71']
for idx, (model_name, model) in enumerate(models.items()):
    fpr, tpr, _ = roc_curve(y_test, probabilities[model_name])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[idx], lw=2, label=f'{model_name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("✓ ROC curves visualized")

In [ ]:
# Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Feature Importance Analysis', fontsize=14, fontweight='bold')

# Random Forest
rf_importance = rf.feature_importances_
rf_indices = np.argsort(rf_importance)[::-1]
axes[0].barh(np.array(feature_names)[rf_indices], rf_importance[rf_indices], color='#3498db')
axes[0].set_title('Random Forest Feature Importance', fontweight='bold')
axes[0].set_xlabel('Importance')

# XGBoost
xgb_importance = xgb.feature_importances_
xgb_indices = np.argsort(xgb_importance)[::-1]
axes[1].barh(np.array(feature_names)[xgb_indices], xgb_importance[xgb_indices], color='#e74c3c')
axes[1].set_title('XGBoost Feature Importance', fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()
print("✓ Feature importance visualized")

## Step 10: Best Model Selection & Summary

In [ ]:
print("\n[STEP 9: BEST MODEL SELECTION]")
print("-" * 100)

best_idx = results_df['ROC-AUC'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
best_model = models[best_model_name]

print(f"\n✓ Best Model: {best_model_name}")
print(f"  - ROC-AUC Score: {results_df.loc[best_idx, 'ROC-AUC']:.4f}")
print(f"  - Accuracy: {results_df.loc[best_idx, 'Accuracy']:.4f}")
print(f"  - F1-Score: {results_df.loc[best_idx, 'F1-Score']:.4f}")

print(f"\n✅ PIPELINE EXECUTION COMPLETE!")
print(f"\n📊 FINAL MODEL PERFORMANCES:")
print(results_df.to_string(index=False))

## Step 11: How to Use the Best Model for Predictions

In [ ]:
print("\n" + "="*100)
print("HOW TO USE THE BEST MODEL FOR PREDICTIONS")
print("="*100)

# Example: Make prediction on new data
print(f"\n🔮 Example Prediction:")
print(f"\nFor a new patient with features:")

# Create example new patient
new_patient = np.array([[45, 70, 14.5, 23.5, 1600, 120, 78, 37.0, 25]])
print(f"HRV=45, Heart_Rate=70, RR_Mean=14.5, BMI=23.5, BMR=1600,")
print(f"Systolic_BP=120, Diastolic_BP=78, Temperature=37.0, Age=25")

# Normalize using the same scaler
new_patient_scaled = scaler.transform(new_patient)

# Make prediction
prediction = best_model.predict(new_patient_scaled)[0]
probability = best_model.predict_proba(new_patient_scaled)[0]

print(f"\n✓ Prediction Result:")
print(f"  - PCOS Status: {'PCOS Positive (1)' if prediction == 1 else 'PCOS Negative (0)'}")
print(f"  - Probability of No PCOS: {probability[0]:.2%}")
print(f"  - Probability of PCOS: {probability[1]:.2%}")

print(f"\n💾 To save the model for later use:")
print(f"\njoblib.dump(best_model, 'best_model.joblib')")
print(f"joblib.dump(scaler, 'scaler.joblib')")

print(f"\n📖 To load and use the model later:")
print(f"\nloaded_model = joblib.load('best_model.joblib')")
print(f"loaded_scaler = joblib.load('scaler.joblib')")
print(f"new_patient_scaled = loaded_scaler.transform(new_patient)")
print(f"prediction = loaded_model.predict(new_patient_scaled)")

## ⚠️ Important Limitations & Ethical Considerations

In [ ]:
print("\n" + "="*100)
print("IMPORTANT LIMITATIONS & ETHICAL CONSIDERATIONS")
print("="*100)

print(f"""
⚠️  SYNTHETIC DATA LIMITATIONS:
   1. Generated data is for TRAINING ROBUSTNESS ONLY
   2. Synthetic samples are NOT actual patients
   3. Cannot capture complex disease interactions
   4. May create unrealistic edge cases
   5. Should NOT be used for clinical validation

⚠️  DATASET LIMITATIONS:
   1. Only 30 real samples (very small)
   2. Augmented to 90 samples total
   3. May have demographic biases
   4. Should NOT be used for clinical deployment
   5. Requires real validation on larger cohorts

✅ APPROPRIATE USE CASES:
   - Academic projects and demonstrations
   - Learning machine learning concepts
   - Baseline for research
   - Teaching tool for biomedical students

❌ NOT APPROPRIATE FOR:
   - Clinical diagnosis (consult endocrinologist)
   - Direct patient care (without FDA approval)
   - Clinical trials (insufficient data)
   - Production deployment (unvalidated)

🏥 MEDICAL DISCLAIMER:
   This model is a SCREENING TOOL ONLY. It cannot diagnose PCOS.
   Clinical diagnosis requires:
   - Medical history review
   - Physical examination
   - Hormone level tests
   - Ultrasound imaging
   - Evaluation by qualified healthcare provider

📋 FOR PRODUCTION USE:
   - Validate on large, real patient dataset (1000+ samples)
   - Multi-center clinical studies
   - Regulatory approval (FDA, CE marking, etc.)
   - Continuous monitoring and retraining
   - Ethical review and compliance
""")

print("="*100)

## 🎓 Summary of What We Accomplished

This notebook implements a **complete ML pipeline** with:

✅ **9 Major Stages:**
1. Data loading and exploration
2. Exploratory data analysis (EDA)
3. Feature normalization (StandardScaler)
4. Synthetic data augmentation (SMOTE + Gaussian noise)
5. Train-test split
6. Model training (3 models)
7. Model evaluation (6 metrics)
8. Result visualization
9. Best model selection

✅ **Comprehensive Evaluation:**
- Accuracy, Precision, Recall, F1-Score
- ROC-AUC, Confusion Matrices
- Feature importance analysis
- Side-by-side model comparison

✅ **Production-Ready Code:**
- Well-organized and documented
- Ready for deployment
- Easy to modify and extend
- Includes example predictions

🎯 **Perfect for:**
- Biomedical engineering projects
- Machine learning courses
- Healthcare AI applications
- Medical informatics learning

---

**Next Steps:**
1. Try with your own dataset
2. Experiment with different models
3. Tune hyperparameters
4. Validate on real patient data
5. Deploy for clinical use (with proper validation and approval)